# 07 · LLM Alias-Review Validation

**Purpose.** Validate the optional **LLM-assisted alias review** layer (`app/src/llm/`) that sits on top of the already-validated PED alias map + tblastn accuracy (unit 06). Two tracks:
- **Track A** — known excluded/unresolved raw names → `review_unresolved_names` (same call as `ui/stages/resolve.py`).
- **Track B** — real uncertain qualifier values from the 100 PED records, gated by the real `needs_llm_review()`, scored by the deterministic classifier, then sent to the LLM.

This notebook is a thin reproducible wrapper over the existing scripts — logic lives in `build_dataset.py` / `run_llm_validation.py`, which are the source of truth.

## Inputs

- `build_dataset.py`, `run_llm_validation.py`, `track_a_ground_truth.py` (this folder)
- PED data reused as ground truth (from unit 06)
- **Real run** needs `OPENAI_API_KEY` in `.env` + network to api.openai.com. Use `--mock` for mechanics without API calls.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "app" / "src").exists():
        ROOT = candidate
        break
UNIT = ROOT / "app" / "validation" / "03_alias_suggestion"
OUT  = UNIT / "outputs"

# "mock" = no API calls (fast, deterministic mechanics); "real" = live LLM (needs OPENAI_API_KEY)
MODE = "mock"

## Run — rebuild dataset + run both tracks

In [ ]:
def run(cmd):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

run([sys.executable, UNIT / "build_dataset.py"])
run([sys.executable, UNIT / "run_llm_validation.py"] + (["--mock"] if MODE == "mock" else []))

## Metrics — action accuracy per track

- **action_accuracy**: LLM chose the correct action (save_alias / skip / ignore).
- **canonical_accuracy_given_save**: when saving, the canonical name is right.
- **dangerous_false_positives**: saved an alias that should have been ignored (the costly error).

In [ ]:
summary = pd.read_csv(OUT / "summary.tsv", sep="\t")
track_a = pd.read_csv(OUT / "track_a_results.tsv", sep="\t")
track_b = pd.read_csv(OUT / "track_b_results.tsv", sep="\t")
summary

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(summary["track"], summary["action_accuracy"] * 100)
for i, v in enumerate(summary["action_accuracy"]):
    ax.text(i, v * 100 + 1, f"{v*100:.0f}%", ha="center")
ax.set_ylabel("action accuracy %"); ax.set_ylim(0, 105)
ax.set_title(f"LLM alias-review accuracy ({MODE})")
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
fig.tight_layout(); fig.savefig(OUT / "llm_review_accuracy.png", dpi=200)
print("dangerous false positives:", summary["dangerous_false_positives"].tolist())

## Interpretation

> ⚠️ **TODO**: báo action_accuracy Track A/B và nhấn dangerous_false_positives = 0 (LLM không tự ý save alias sai). Chạy lại với MODE='real' để lấy số chính thức cho bài; số mock chỉ chứng minh cơ chế.